In [ ]:
!pip install llama-cpp-python \
  --prefer-binary \
  --extra-index-url=https://jllllll.github.io/llama-cpp-python-cuBLAS-wheels/AVX2/cu121

Looking in indexes: https://pypi.org/simple, https://jllllll.github.io/llama-cpp-python-cuBLAS-wheels/AVX2/cu121


In [ ]:
!pip install pandas sentence-transformers faiss-cpu tqdm
!pip install -q sentence-transformers faiss-cpu pandas

In [ ]:
!wget https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.1-GGUF/resolve/main/mistral-7b-instruct-v0.1.Q4_K_M.gguf

--2025-05-02 14:50:12--  https://huggingface.co/TheBloke/Mistral-7B-Instruct-v0.1-GGUF/resolve/main/mistral-7b-instruct-v0.1.Q4_K_M.gguf
Resolving huggingface.co (huggingface.co)... 18.164.174.55, 18.164.174.23, 18.164.174.118, ...
Connecting to huggingface.co (huggingface.co)|18.164.174.55|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://cdn-lfs.hf.co/repos/46/12/46124cd8d4788fd8e0879883abfc473f247664b987955cc98a08658f7df6b826/14466f9d658bf4a79f96c3f3f22759707c291cac4e62fea625e80c7d32169991?response-content-disposition=inline%3B+filename*%3DUTF-8%27%27mistral-7b-instruct-v0.1.Q4_K_M.gguf%3B+filename%3D%22mistral-7b-instruct-v0.1.Q4_K_M.gguf%22%3B&Expires=1746201012&Policy=eyJTdGF0ZW1lbnQiOlt7IkNvbmRpdGlvbiI6eyJEYXRlTGVzc1RoYW4iOnsiQVdTOkVwb2NoVGltZSI6MTc0NjIwMTAxMn19LCJSZXNvdXJjZSI6Imh0dHBzOi8vY2RuLWxmcy5oZi5jby9yZXBvcy80Ni8xMi80NjEyNGNkOGQ0Nzg4ZmQ4ZTA4Nzk4ODNhYmZjNDczZjI0NzY2NGI5ODc5NTVjYzk4YTA4NjU4ZjdkZjZiODI2LzE0NDY2ZjlkNjU4YmY0YTc5Zjk2YzNmM2Yy

In [ ]:
from llama_cpp import Llama
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [ ]:
llm = Llama(
    model_path="mistral-7b-instruct-v0.1.Q4_K_M.gguf",
    n_ctx=2480,  # Context window
    n_gpu_layers=-1,  # Try increasing layers on GPU
    n_threads=16,  # Using all available cores
    n_batch=32
)

AVX = 1 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


In [ ]:
!nvidia-smi

Fri May  2 14:51:08 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   67C    P0             29W /   70W |    9118MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Load your dataset
df = pd.read_json("Excuis_AI.json")

# Combine relevant fields into a single text for embedding
df["text_for_embedding"] = (
    "Dish: " + df["name"] +
    "; Ingredients: " + df["ingredients"].apply(", ".join) +
    "; Region: " + df["region"]
)
# Convert all to string and fill NaNs with empty string
df["text_for_embedding"] = df["text_for_embedding"].fillna("").astype(str)

In [ ]:
# Load a lightweight embedding model (no GPU needed)
model = SentenceTransformer('all-MiniLM-L6-v2')  # 384-dimensional embeddings

# Generate embeddings for all recipes
embeddings = model.encode(df["text_for_embedding"].tolist(), show_progress_bar=True)

# Convert to FAISS index for fast search
index = faiss.IndexFlatL2(embeddings.shape[1])
index.add(embeddings)  # Add embeddings to the index

Batches:   0%|          | 0/81 [00:00<?, ?it/s]

In [ ]:
def retrieve_recipes(query, top_k=3):
    # Embed the user query
    query_embedding = model.encode([query])

    # Search FAISS index
    distances, indices = index.search(query_embedding, top_k)

    # Return matched recipes
    return df.iloc[indices[0]].to_dict("records")

In [ ]:
def clean_dish_name(name):
    # Simplify name by removing repeated words and chopping at "recipe"
    name = name.lower().split("recipe")[0]
    words = name.split()
    seen = set()
    cleaned = [word for word in words if word not in seen and not seen.add(word)]
    return " ".join(cleaned).title()

In [ ]:
def generate_response(user_query, matched_recipes):
    context = "\n".join(
        f"Dish: {r['name']}\nIngredients: {r['ingredients']}\nInstructions: {r['instructions'][:200]}..."
        for r in matched_recipes
    )

    prompt = f"""[INST] <<SYS>>
You are a professional recipe assistant. Only respond with recipes listed below.
Always format your answer clearly with headings: "Ingredients" and "Instructions".
If a dish is not in the list, say: "Sorry, I don't have a recipe for that."
<</SYS>>
{context}
Question: {user_query}[/INST]
"""
    response = llm(
        prompt,
        max_tokens=1500,
        temperature=0.3,
        stop=["</s>"]  # Mistral's stop token
    )
    return response['choices'][0]['text']


In [ ]:
!pip install gradio

In [ ]:
import gradio as gr

# Chat function
def chat_interface(user_input, history):
    if user_input.lower() in ["exit", "quit"]:
        return "Goodbye! 👋", history

    matched = retrieve_recipes(user_input)
    if not matched:
        bot_reply = "I couldn't find any matching recipes."
    else:
        bot_reply = generate_response(user_input, matched)

    history.append((user_input, bot_reply))
    return "", history

background_url = "https://images.pexels.com/photos/1640773/pexels-photo-1640773.jpeg"

custom_css = f"""
.gradio-container {{
    background: url('{background_url}') no-repeat center center fixed;
    background-size: cover;
    font-family: 'Segoe UI', sans-serif;
}}

/* Chatbot container styling */
.chatbox {{
    backdrop-filter: blur(10px);
    background-color: rgba(0, 0, 0, 0.6) !important;  /* Dark semi-transparent background */
    border-radius: 20px;
    padding: 15px;
    box-shadow: 0 8px 32px 0 rgba(31, 38, 135, 0.37);
    color: white !important;  /* Light text color */
}}

/* Message bubbles styling */
.message {{
    padding: 10px 15px;
    border-radius: 18px;
    margin: 5px 0;
    max-width: 80%;
}}

.user-message {{
    background-color: rgba(75, 85, 99, 0.8) !important;  /* Darker gray for user */
    color: white !important;
    margin-left: auto;
    border-bottom-right-radius: 5px !important;
}}

.bot-message {{
    background-color: rgba(55, 65, 81, 0.9) !important;  /* Slightly lighter gray for bot */
    color: white !important;
    margin-right: auto;
    border-bottom-left-radius: 5px !important;
}}

/* Title styling */
.title {{
    color: rgba(255, 255, 255, 0.9) !important;
    text-shadow: 2px 2px 4px rgba(0, 0, 0, 0.7) !important;
    font-weight: 600 !important;
    font-size: 1.5rem !important;
    margin-left: 10px !important;
}}

/* Logo styling */
.logo-image {{
    max-width: 80px !important;
    max-height: 80px !important;
    object-fit: contain;
    border-radius: 8px;
    border: 2px solid rgba(255, 255, 255, 0.2);
}}

/* Textbox styling */
.textbox {{
    background-color: rgba(0, 0, 0, 0.5) !important;
    color: white !important;
    border-radius: 12px !important;
}}
"""

with gr.Blocks(css=custom_css) as demo:
    with gr.Row():
        gr.Image(
            value="Excuis AI logo.png",
            width=80,
            height=80,
            show_label=False,
            show_download_button=False,
            elem_classes="logo-image"
        )
        gr.Markdown("## 🧑‍🍳 Excuis AI — Recipe Assistant", elem_classes="title")

    chatbot = gr.Chatbot(
        label="",
        height=400,
        elem_classes="chatbox",
        bubble_full_width=False
    )

    user_input = gr.Textbox(
        label="Type your question...",
        placeholder="e.g. How to make Mango Shake?",
        elem_classes="textbox"
    )

    clear = gr.Button("Clear Chat", variant="secondary")

    user_input.submit(chat_interface, [user_input, chatbot], [user_input, chatbot])
    clear.click(lambda: [], None, chatbot)

demo.launch(share=True)


<ipython-input-44-e7cb24064203>:96: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(
<ipython-input-44-e7cb24064203>:96: DeprecationWarning: The 'bubble_full_width' parameter is deprecated and will be removed in a future version. This parameter no longer has any effect.
  chatbot = gr.Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2afbeeff31526b75af.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


###Extras

In [ ]:
'''import gradio as gr
def chat(user_input, history):
    matched = retrieve_recipes(user_input)
    if not matched:
        return history + [[user_input, "Sorry, I couldn't find any matching recipes."]]
    response = generate_response(user_input, matched)
    return history + [[user_input, response]]

# Your background image URL
background_image_url = "https://images.unsplash.com/photo-1498579809087-ef1e558fd1da"

# Define custom CSS for blurred chatbox
custom_css = f"""
body {{
    background: url('{background_image_url}') no-repeat center center fixed;
    background-size: cover;
}}

.chatbox {{
    backdrop-filter: blur(10px);
    background-color: rgba(255, 255, 255, 0.3);
    border-radius: 20px;
    padding: 20px;
    box-shadow: 0 8px 32px 0 rgba(31, 38, 135, 0.37);
}}
"""

with gr.Blocks(css=custom_css) as demo:
    gr.Markdown("# 🍽️ Excuis AI — Your Recipe Assistant", elem_classes="chatbox")

    chatbot = gr.Chatbot(elem_classes="chatbox")
    msg = gr.Textbox(label="Type your recipe query...", placeholder="e.g., I need a recipe for Mango Shake")
    clear = gr.Button("Clear Chat")

    state = gr.State([])

    msg.submit(chat, [msg, state], [chatbot, state])
    clear.click(lambda: ([], []), None, [chatbot, state])

demo.launch()'''


<ipython-input-13-9b00253790c9>:31: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(elem_classes="chatbox")


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5183ca4413a63b3ca6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
'''while True:
    user_input = input("\nYou: ")
    if user_input.lower() in ["exit", "quit"]:
        break

    # Retrieve relevant recipes
    matched_recipes = retrieve_recipes(user_input)

    if not matched_recipes:
        print("Excuis AI: I couldn't find any matching recipes.")
        continue

    # Generate response
    response = generate_response(user_input, matched_recipes)
    print(f"\nExcuis AI: {response}")'''

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/gradio/queueing.py", line 625, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/gradio/route_utils.py", line 322, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/gradio/blocks.py", line 2156, in process_api
    data = await self.postprocess_data(block_fn, result["prediction"], state)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/gradio/blocks.py", line 1890, in postprocess_data
    self.validate_outputs(block_fn, predictions)  # type: ignore
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/gradio/blocks.py", line 1845, in validate_outputs
    rai